In [7]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from nbaPlayerLogs import NBAGameLogs
from pathlib import Path
import sys
import os

# project_root = Path.cwd().parent.parent
# project_root_str = str(project_root)

# if project_root_str not in sys.path:
#     sys.path.insert(0, project_root_str)

# os.chdir(project_root_str)

pd.set_option('display.max_columns', None)

## Fetches Player Gamelogs

In [2]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2025-26', 
#     season_type='Regular Season', 
#     sleep_time=2, 
#     max_workers=5,
#     batch_limit=100,
#     complete_cache_file='data/raw/season_stats/S26.csv',
#     include_playbyplay=False
# )
# data.tail()

In [9]:
# df = NBAGameLogs(season='2025-26', season_type='Regular Season').fetch(skip_start_positions=True).build().get_df()
# df.head()

# Session by session — run this repeatedly until all games are done
logs = NBAGameLogs(season='2025-26', season_type='Regular Season')
logs.fetch(batch_size=100, checkpoint_path='tracking_checkpoint.csv')

Fetching data for 2025-26 Regular Season...
✓ Player base
✓ Player advanced
✓ Team base
✓ Team advanced
✓ Checkpoint loaded — 1246/1230 games already done, 0 remaining
✓ All games already fetched from checkpoint
✓ START_POSITION ready (32630 rows)


In [2]:
"""
BettingPros NBA Prop Bets Scraper
Fetches each market separately to get all props.
"""

import requests
import json
import csv
from datetime import date

BASE_URL = "https://api.bettingpros.com/v3/props"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/121.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.bettingpros.com",
    "Referer": "https://www.bettingpros.com/",
}

MARKET_NAMES = {
    156: "Points",
    151: "Assists",
    157: "Rebounds",
    335: "Pts+Ast",
    336: "Pts+Reb",
    337: "Reb+Ast",
    338: "Pts+Reb+Ast",
    152: "Steals",
    160: "Blocks",
    162: "3-Pointers Made",
}


def fetch_market(target_date: str, market_id: int, limit: int = 100, offset: int = 0) -> dict:
    params = {
        "limit": limit,
        "offset": offset,
        "sport": "NBA",
        "market_id": market_id,
        "date": target_date,
        "include_selections": "false",
        "include_filter_graphs": "false",
        "data_points": 8,
        "min_odds": -1000,
        "max_odds": 1000,
        "ev_threshold_min": -0.4,
        "ev_threshold_max": 0.4,
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def parse_props(data: dict) -> list[dict]:
    rows = []
    for prop in data.get("props", []):
        proj = prop.get("projection") or {}
        rows.append({
            "player": prop.get("participant", {}).get("name", "Unknown"),
            "prop":   MARKET_NAMES.get(prop.get("market_id"), prop.get("market_id")),
            "line":   prop.get("over", {}).get("line"),
            "proj":   proj.get("value"),
            "side":   proj.get("recommended_side"),
            "diff":   proj.get("diff"),
        })
    return rows


def scrape_all(target_date: str) -> list[dict]:
    all_rows = []

    print(f"Fetching props for {target_date}...")
    for market_id, market_name in MARKET_NAMES.items():
        seen   = set()
        offset = 0

        while True:
            data = fetch_market(target_date, market_id, limit=500, offset=offset)
            rows = parse_props(data)

            if not rows:
                break

            new_rows = []
            for r in rows:
                key = (r["player"], r["prop"], r["line"])
                if key not in seen:
                    seen.add(key)
                    new_rows.append(r)

            if not new_rows:
                break

            all_rows.extend(new_rows)
            offset += 100

        print(f"  {market_name:<20} → {len([r for r in all_rows if r['prop'] == market_name])} props")

    print(f"\n  Total: {len(all_rows)} props")
    return all_rows


def save_csv(rows: list[dict], filename: str):
    if not rows:
        print("No data to save.")
        return
    fieldnames = list(rows[0].keys())
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} rows → {filename}")


def save_json(rows: list[dict], filename: str):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    print(f"Saved {len(rows)} rows → {filename}")


from datetime import date, timedelta

START_DATE    = date(2025, 4, 1)
END_DATE      = date(2025, 4, 15)
OUTPUT_FORMAT = "csv"
OUTPUT_NAME   = "nba_props"

current = START_DATE
while current <= END_DATE:
    target = str(current)
    print(f"\n{'='*50}\nProcessing {target}\n{'='*50}")
    
    rows = scrape_all(target)
    
    if rows:
        if OUTPUT_FORMAT in ("csv", "both"):
            save_csv(rows, f"historical_odds/{target}.csv")
        if OUTPUT_FORMAT in ("json", "both"):
            save_json(rows, f"historical_odds/{target}.json")
    else:
        print(f"  No props found for {target}, skipping.")
    
    current += timedelta(days=1)

print("\nDone!")


Processing 2025-04-01
Fetching props for 2025-04-01...
  Points               → 76 props
  Assists              → 81 props
  Rebounds             → 78 props
  Pts+Ast              → 75 props
  Pts+Reb              → 73 props
  Reb+Ast              → 73 props
  Pts+Reb+Ast          → 72 props
  Steals               → 76 props
  Blocks               → 83 props
  3-Pointers Made      → 76 props

  Total: 763 props


FileNotFoundError: [Errno 2] No such file or directory: 'historical_odds/2025-04-01.csv'